# Feature Engineering & Preprocessing

This notebook prepares the AISI-4140 heat treatment dataset for modeling. It loads the raw data, inspects and cleans it, splits it into features and targets, and builds a `scikit-learn` preprocessing pipeline (scaling numerical features and one-hot encoding categorical features). The fitted pipeline is saved so it can be reused during model training and inference.

## 1. Importing Modules and Loading the Dataset

Mount Google Drive so the dataset stored there can be accessed, then import the libraries needed for data handling and preprocessing.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load the AISI-4140 heat treatment dataset from Google Drive and check its shape.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#Loading Dataset

df = pd.read_csv(
    "/content/drive/MyDrive/Heat_Treatment_Dashboard/data/aisi_4140_heat_treatment_dataset.csv"
)

print("Dataset shape:",df.shape)

Dataset shape: (1100, 11)


## 2. Initial Data Inspection

Create a working copy of the dataset and check its structure, data types, missing values, and duplicate rows.

In [ ]:
df_ml = df.copy()

print(df_ml.shape)

(1100, 11)


Check the column data types and non-null counts.

In [ ]:
df_ml.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   austenitizing_temp_C   1100 non-null   float64
 1   section_thickness_mm   1100 non-null   float64
 2   quench_medium          1100 non-null   object 
 3   tempering_temp_C       1100 non-null   float64
 4   tempering_time_hr      1100 non-null   float64
 5   hardness_HRC           1100 non-null   float64
 6   UTS_MPa                1100 non-null   float64
 7   YS_MPa                 1100 non-null   float64
 8   elongation_pct         1100 non-null   float64
 9   reduction_of_area_pct  1100 non-null   float64
 10  source                 1100 non-null   object 
dtypes: float64(9), object(2)
memory usage: 94.7+ KB


Check for missing values in each column.

In [ ]:
df.isnull().sum()

,0
austenitizing_temp_C,0
section_thickness_mm,0
quench_medium,0
tempering_temp_C,0
tempering_time_hr,0
hardness_HRC,0
UTS_MPa,0
YS_MPa,0
elongation_pct,0
reduction_of_area_pct,0


Check for duplicate rows.

No missing values or duplicate rows are found, so no imputation or deduplication is required.

In [ ]:
df_ml.duplicated().sum()

np.int64(0)

## 3. Dropping Non-Predictive Columns

The `source` column only records where a data point came from and is not a physical process parameter, so it is dropped before modeling.

In [ ]:
df_ml = df_ml.drop(columns=["source"])
df_ml.head()

,austenitizing_temp_C,section_thickness_mm,quench_medium,tempering_temp_C,tempering_time_hr,hardness_HRC,UTS_MPa,YS_MPa,elongation_pct,reduction_of_area_pct
0,845.0,25.0,oil,815.0,1.0,13.0,655.0,415.0,25.7,56.9
1,845.0,25.0,oil,316.0,1.0,47.0,1551.0,1372.6,12.0,42.0
2,845.0,25.0,oil,538.0,1.0,26.0,896.0,672.0,18.0,55.0
3,851.7,50.0,oil,384.7,4.0,38.8,1212.1,1004.6,17.1,44.3
4,834.4,100.0,air,401.7,4.0,37.4,1177.4,952.6,19.3,44.9


## 4. Defining Feature and Target Variables

**Features (`X`)** are the heat treatment process parameters (temperatures, thickness, quench medium, tempering time).

In [ ]:
feature_columns = [
    "austenitizing_temp_C",
    "section_thickness_mm",
    "quench_medium",
    "tempering_temp_C",
    "tempering_time_hr",
]

X = df_ml[feature_columns]
X.head()

,austenitizing_temp_C,section_thickness_mm,quench_medium,tempering_temp_C,tempering_time_hr
0,845.0,25.0,oil,815.0,1.0
1,845.0,25.0,oil,316.0,1.0
2,845.0,25.0,oil,538.0,1.0
3,851.7,50.0,oil,384.7,4.0
4,834.4,100.0,air,401.7,4.0


**Targets (`y`)** are the resulting mechanical properties to be predicted: hardness, tensile strength, yield strength, elongation, and reduction of area.

In [ ]:
# target Variables
target_columns = [
    "hardness_HRC",
    "UTS_MPa",
    "YS_MPa",
    "elongation_pct",
    "reduction_of_area_pct"
]
y = df_ml[target_columns]
y.head()

,hardness_HRC,UTS_MPa,YS_MPa,elongation_pct,reduction_of_area_pct
0,13.0,655.0,415.0,25.7,56.9
1,47.0,1551.0,1372.6,12.0,42.0
2,26.0,896.0,672.0,18.0,55.0
3,38.8,1212.1,1004.6,17.1,44.3
4,37.4,1177.4,952.6,19.3,44.9


Confirm the shapes of the feature matrix and target matrix line up.

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1100, 5)
y shape: (1100, 5)


## 5. Categorizing Features

Separate the features into **numerical** columns (to be scaled) and **categorical** columns (to be one-hot encoded).

In [ ]:
# Numerical features

numerical_features = [
    "austenitizing_temp_C",
    "section_thickness_mm",
    "tempering_temp_C",
    "tempering_time_hr"
]

# Categorical features

categorical_features = [
    "quench_medium"
]

## 6. Train-Test Split

Split the data into training (80%) and testing (20%) sets, using a fixed random state for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

Confirm the number of samples in each split.

In [ ]:
print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

Training samples: 880
Testing samples : 220


## 7. Building Preprocessing Pipelines

Define a pipeline for numerical features that standardizes them (zero mean, unit variance).

In [ ]:
# Numerical preprocessing

numerical_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

Define a pipeline for categorical features that one-hot encodes them, ignoring any unseen categories at inference time.

In [ ]:
# Categorical preprocessing

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

Combine both pipelines into a single `ColumnTransformer` that applies the right transformation to each group of columns.

In [ ]:
# Combine numerical and categorical preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 8. Applying Preprocessing to the Train and Test Sets

Fit the preprocessor on the training data only, then use it to transform both the training and test sets (preventing data leakage from the test set).

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

Verify the shapes before and after transformation.

In [ ]:
print("Original X_train shape:", X_train.shape)
print("Processed X_train shape:", X_train_processed.shape)

print("Original X_test shape:", X_test.shape)
print("Processed X_test shape:", X_test_processed.shape)

Original X_train shape: (880, 5)
Processed X_train shape: (880, 7)
Original X_test shape: (220, 5)
Processed X_test shape: (220, 7)


Inspect the generated feature names, including the one-hot encoded quench medium categories.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)

['num__austenitizing_temp_C' 'num__section_thickness_mm'
 'num__tempering_temp_C' 'num__tempering_time_hr' 'cat__quench_medium_air'
 'cat__quench_medium_oil' 'cat__quench_medium_water']


## 9. Inspecting the Transformed Feature Set

Convert the processed training features back into a DataFrame (with proper column names) for easier inspection.

In [ ]:
X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df.head()

,num__austenitizing_temp_C,num__section_thickness_mm,num__tempering_temp_C,num__tempering_time_hr,cat__quench_medium_air,cat__quench_medium_oil,cat__quench_medium_water
507,0.867732,-0.916580,0.204543,0.127705,0.0,0.0,1.0
551,-0.122666,0.807984,0.093347,-0.999854,0.0,1.0,0.0
290,1.050574,-0.916580,1.088096,-0.624001,0.0,1.0,0.0
2,-0.404548,-0.916580,0.896509,-0.624001,0.0,1.0,0.0
6,1.599102,-0.341725,-1.252268,-0.624001,0.0,0.0,1.0


Check the data types and structure of the processed feature set.

In [ ]:
X_train_processed_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 880 entries, 507 to 860
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   num__austenitizing_temp_C  880 non-null    float64
 1   num__section_thickness_mm  880 non-null    float64
 2   num__tempering_temp_C      880 non-null    float64
 3   num__tempering_time_hr     880 non-null    float64
 4   cat__quench_medium_air     880 non-null    float64
 5   cat__quench_medium_oil     880 non-null    float64
 6   cat__quench_medium_water   880 non-null    float64
dtypes: float64(7)
memory usage: 55.0 KB


Review summary statistics to confirm the numerical features are properly scaled.

In [ ]:
X_train_processed_df.describe()

,num__austenitizing_temp_C,num__section_thickness_mm,num__tempering_temp_C,num__tempering_time_hr,cat__quench_medium_air,cat__quench_medium_oil,cat__quench_medium_water
count,8.800000e+02,8.800000e+02,8.800000e+02,8.800000e+02,880.000000,880.000000,880.000000
mean,6.209679e-15,3.835316e-17,1.332268e-16,-2.321375e-17,0.109091,0.697727,0.193182
std,1.000569e+00,1.000569e+00,1.000569e+00,1.000569e+00,0.311931,0.459504,0.395019
min,-1.699682e+00,-1.192510e+00,-1.624172e+00,-9.998542e-01,0.000000,0.000000,0.000000
25%,-8.845095e-01,-9.165797e-01,-9.246923e-01,-9.998542e-01,0.000000,0.000000,0.000000
50%,-1.600738e-02,-3.417250e-01,-2.911795e-02,-6.240013e-01,0.000000,1.000000,0.000000
75%,8.677316e-01,8.079844e-01,8.544353e-01,1.277046e-01,0.000000,1.000000,0.000000
max,1.644812e+00,1.957694e+00,2.977668e+00,1.631116e+00,1.000000,1.000000,1.000000


## 10. Target Variable Statistics

Summary statistics for the training set targets.

In [ ]:
y_train.describe()

,hardness_HRC,UTS_MPa,YS_MPa,elongation_pct,reduction_of_area_pct
count,880.000000,880.000000,880.000000,880.000000,880.000000
mean,36.447614,1221.333977,978.172614,18.409659,43.031250
std,11.157874,339.411702,355.810715,4.961959,9.264065
min,13.000000,655.000000,415.000000,8.700000,24.900000
25%,26.075000,908.825000,643.525000,13.900000,34.775000
50%,37.500000,1203.550000,947.450000,18.600000,43.800000
75%,47.125000,1539.200000,1312.650000,22.900000,51.200000
max,54.500000,1923.300000,1663.900000,28.400000,61.800000


Summary statistics for the test set targets.

In [ ]:
y_test.describe()

,hardness_HRC,UTS_MPa,YS_MPa,elongation_pct,reduction_of_area_pct
count,220.000000,220.000000,220.000000,220.000000,220.000000
mean,36.181364,1207.990000,971.676818,18.495909,43.048636
std,11.443611,344.279893,360.877170,5.013164,9.529754
min,17.300000,694.300000,482.600000,9.200000,26.600000
25%,24.875000,880.025000,632.725000,13.775000,34.100000
50%,36.400000,1179.550000,931.850000,18.800000,43.450000
75%,47.075000,1543.125000,1319.250000,22.925000,51.625000
max,53.800000,1852.300000,1687.600000,27.300000,64.300000


## 11. Saving the Preprocessing Pipeline

Persist the fitted `preprocessor` to disk with `joblib` so the exact same transformations can be reused later for model training and inference.

In [ ]:
import joblib

joblib.dump(
    preprocessor,
    "/content/drive/MyDrive/Heat_Treatment_Dashboard/models/preprocessor.pkl"
)

['/content/drive/MyDrive/Heat_Treatment_Dashboard/models/preprocessor.pkl']